# Image Style Transfer
"Image Style Transfer Using Convolutional Neural Networks" (Gatys, Ecker, Bethge, 2016)

**Idea:** a CNN trained for classification separates *what* is in an image from *how* it is painted. Deep feature maps keep the layout of objects (**content**); the correlations between feature channels, with positions thrown away, capture textures and brush strokes (**style**). Start from any image and run gradient descent **on its pixels** until its deep features match a photo and its feature correlations match a painting.

### What is being optimized
| | Ordinary training (`convolution.ipynb`) | Style transfer (this notebook) | Fast style transfer (Johnson et al., 2016) |
|---|---|---|---|
| Optimized variable | network weights | **the image's pixels** | weights of a feed-forward stylizing net |
| Network weights | learned | **frozen** (pretrained VGG-19) | VGG frozen (used as the loss) |
| Loss compares | prediction vs label | VGG features of the image vs content / style targets | same perceptual loss |
| Data needed | labeled dataset | **2 images** | an image dataset + 1 style |
| Cost per result | one forward pass | hundreds of forward + backward passes | one forward pass |

Here: VGG-19 features, content loss, Gram-matrix style loss, optimized with L-BFGS as in the paper.

Shape flow: `image (1, 3, H, W) -> VGG-19 conv layers -> features F_l (1, C_l, H_l, W_l) -> content: F at conv4_2 | style: Gram G_l = F_l F_l^T (C_l, C_l) at conv1_1 .. conv5_1 -> loss (scalar) -> gradient w.r.t. the image`

In [ ]:
import os
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Images
- **Content**: a photo from Imagenette (already in `../data`, used in `image_net.ipynb`).
- **Style**: Van Gogh's *The Starry Night* (public domain), downloaded once into `../data`. If the download fails, another Imagenette photo is used so the notebook still runs.

Both are resized to the same size and normalized with the ImageNet mean / std that VGG was trained with.

In [ ]:
content_path = "../data/imagenette2-320/val/n03028079/ILSVRC2012_val_00003351.JPEG"   # a church
style_path = "../data/starry_night.jpg"
style_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/960px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"
fallback_style = "../data/imagenette2-320/val/n03888257/" + sorted(os.listdir("../data/imagenette2-320/val/n03888257"))[0]

if not os.path.exists(style_path):
    try:
        req = urllib.request.Request(style_url, headers={"User-Agent": "style-transfer-notebook/1.0"})
        with urllib.request.urlopen(req, timeout=30) as r, open(style_path, "wb") as f:
            f.write(r.read())
    except Exception as e:
        print("style download failed, using an Imagenette photo instead:", e)
        style_path = fallback_style

img_size = 384   # lower this if the GPU runs out of memory
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def load(path):
    # -> (1, 3, H, W) in [0, 1]
    tf = transforms.Compose([transforms.Resize(img_size), transforms.CenterCrop(img_size), transforms.ToTensor()])
    return tf(Image.open(path).convert("RGB"))[None].to(device)

def show(imgs, titles):
    fig, axes = plt.subplots(1, len(imgs), figsize=(5 * len(imgs), 5))
    for ax, im, t in zip(axes, imgs, titles):
        ax.imshow(im[0].detach().clamp(0, 1).permute(1, 2, 0).cpu()); ax.set_title(t); ax.axis("off")
    plt.show()

content_img, style_img = load(content_path), load(style_path)
print(content_img.shape, style_img.shape)
show([content_img, style_img], ["content", "style"])

## 2. Feature extractor - VGG-19
The convolutional part of VGG-19 (16 conv layers in 5 blocks, see the VGG section of `image_net.ipynb`), pretrained on ImageNet and **frozen**. The classifier head is dropped, so any image size works.

Two changes from the paper:
- max pooling -> **average pooling** (smoother gradients, slightly nicer results)
- features are taken **after the ReLU** of each named conv layer

Layers used (paper's choice):
- content: `conv4_2` - deep enough to ignore exact pixel values, shallow enough to keep spatial layout
- style: `conv1_1, conv2_1, conv3_1, conv4_1, conv5_1` - from fine textures (small receptive field) to large strokes

In [ ]:
content_layers = ["conv4_2"]
style_layers = ["conv1_1", "conv2_1", "conv3_1", "conv4_1", "conv5_1"]

class VGGFeatures(nn.Module):
    # image (1, 3, H, W) in [0, 1] -> {layer name: features (1, C_l, H_l, W_l)}
    def __init__(self, layers):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features
        self.layers = set(layers)
        self.blocks, self.names = nn.ModuleList(), []
        block, conv = 1, 0
        for m in vgg:
            if isinstance(m, nn.Conv2d):
                conv += 1
                self.names.append(f"conv{block}_{conv}")
            elif isinstance(m, nn.ReLU):
                m = nn.ReLU(inplace=False)                 # in-place would overwrite the features we keep
                self.names.append(f"relu{block}_{conv}")
            else:
                m = nn.AvgPool2d(2)                        # replaces MaxPool2d
                self.names.append(f"pool{block}")
                block, conv = block + 1, 0
            self.blocks.append(m)
        last = max(i for i, n in enumerate(self.names) if n.replace("relu", "conv") in self.layers and n.startswith("relu"))
        self.blocks, self.names = self.blocks[:last + 1], self.names[:last + 1]   # nothing after conv5_1 is needed
        for p in self.parameters():
            p.requires_grad_(False)

    def forward(self, x):
        x = (x - mean) / std
        out = {}
        for name, m in zip(self.names, self.blocks):
            x = m(x)
            if name.startswith("relu") and name.replace("relu", "conv") in self.layers:
                out[name.replace("relu", "conv")] = x
        return out

extractor = VGGFeatures(content_layers + style_layers).to(device).eval()
with torch.no_grad():
    for name, f in extractor(content_img).items():
        print(f"{name}: {tuple(f.shape)}")

## 3. Content loss
The generated image should have the same deep features as the photo, position by position.

### Formula: Content loss
$$\mathcal{L}_{content} = \frac{1}{C_l H_l W_l} \sum_{c, i, j} \big(F^l_{cij}(x) - F^l_{cij}(p)\big)^2$$
- $x$: the image being optimized; $p$: the content photo
- $F^l(\cdot) \in \mathbb{R}^{C_l \times H_l \times W_l}$: feature maps at layer $l$ = `conv4_2`
- a mean squared error in feature space (the paper uses $\frac{1}{2}\sum$; the normalization only rescales the weight)
- two images with the same `conv4_2` features show the same objects in the same places, but can differ in color and texture

## 4. Style loss
Style is "which features fire **together**", regardless of where. The **Gram matrix** of a layer holds the inner product between every pair of channels, summed over all positions, so spatial arrangement is discarded.

### Formula: Gram matrix
$$G^l_{cd} = \frac{1}{C_l H_l W_l} \sum_{i, j} F^l_{cij}\, F^l_{dij}, \qquad G^l = \frac{1}{C_l H_l W_l}\, F^l (F^l)^T \in \mathbb{R}^{C_l \times C_l}$$
- $F^l$ reshaped to $(C_l, H_l W_l)$: one row per channel
- $G^l_{cd}$: correlation between channel $c$ and channel $d$ (e.g. "blue swirls" co-occurring with "yellow blobs")
- the Gram of a `(64, H, W)` map is `(64, 64)` whatever the image size

### Formula: Style loss
$$\mathcal{L}_{style} = \sum_{l} w_l\, \frac{1}{C_l^2} \sum_{c, d} \big(G^l_{cd}(x) - G^l_{cd}(a)\big)^2$$
- $a$: the style image; $l$ runs over the five style layers, $w_l = 1/5$
- matching Grams at several depths matches textures at several scales

### Formula: Total loss
$$\mathcal{L}(x) = \alpha\, \mathcal{L}_{content}(x, p) + \beta\, \mathcal{L}_{style}(x, a)$$
- only the ratio $\alpha / \beta$ matters: small -> more painterly, large -> closer to the photo
- with the normalizations above, $\alpha = 1$, $\beta = 10^6$ gives a balanced result

In [ ]:
def gram(f):
    # (1, C, H, W) -> (C, C)
    _, C, H, W = f.shape
    f = f.reshape(C, H * W)                       # one row per channel
    return f @ f.T / (C * H * W)

def content_loss(feats, targets):
    return sum(F.mse_loss(feats[l], targets[l]) for l in content_layers)

def style_loss(feats, targets):
    return sum(F.mse_loss(gram(feats[l]), targets[l]) for l in style_layers) / len(style_layers)

with torch.no_grad():
    content_targets = {l: f for l, f in extractor(content_img).items() if l in content_layers}
    style_targets = {l: gram(f) for l, f in extractor(style_img).items() if l in style_layers}
print({l: tuple(g.shape) for l, g in style_targets.items()})

## 5. Optimizing the image
The image itself is the only `Parameter`. Initialized from the content photo (converges faster than from noise and keeps the structure).

The paper uses **L-BFGS**, a quasi-Newton method that builds a curvature estimate from recent gradients. It suits this problem: a deterministic, full-batch loss over a few hundred thousand variables. It needs a `closure` that re-evaluates the loss, since each step tries several points along the search direction. (Adam with lr 0.02 also works, with more steps.)

In [ ]:
def stylize(content_weight=1.0, style_weight=1e6, steps=300, init="content", log_every=50):
    x = (content_img.clone() if init == "content" else torch.rand_like(content_img)).requires_grad_(True)   # (1, 3, H, W)
    optimizer = torch.optim.LBFGS([x], lr=1.0, max_iter=steps, history_size=50)
    state = {"i": 0, "log": []}

    def closure():
        optimizer.zero_grad()
        feats = extractor(x)
        lc, ls = content_loss(feats, content_targets), style_loss(feats, style_targets)
        loss = content_weight * lc + style_weight * ls
        loss.backward()
        state["i"] += 1
        state["log"].append((lc.item(), ls.item()))
        if state["i"] % log_every == 0:
            print(f"eval {state['i']:4d} | content {lc.item():.4f} | style {ls.item():.3e} | total {loss.item():.4f}")
        return loss

    optimizer.step(closure)          # runs up to `steps` loss evaluations internally
    return x.detach().clamp(0, 1), state["log"]

result, log = stylize()
show([content_img, style_img, result], ["content", "style", "result"])

lc, ls = zip(*log)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(lc); ax[0].set_title("content loss"); ax[0].set_xlabel("loss evaluation")
ax[1].semilogy(ls); ax[1].set_title("style loss"); ax[1].set_xlabel("loss evaluation")
plt.show()

## 6. The content / style trade-off
Same images, three style weights. With a small $\beta$ the photo is barely touched; with a large one the layout starts to dissolve into texture.

In [ ]:
weights = [1e4, 1e6, 1e8]
results = [stylize(style_weight=w, log_every=10 ** 9)[0] for w in weights]
show(results, [f"style weight {w:.0e}" for w in weights])

## 7. Texture only
With the content weight at 0 and a random start, only the Gram matrices are matched. The result is a **texture** with the painting's colors and strokes and no layout at all, which shows that the Gram matrix really does discard position (this is Gatys et al.'s earlier texture-synthesis paper).

In [ ]:
texture, _ = stylize(content_weight=0.0, init="noise", log_every=10 ** 9)
show([style_img, texture], ["style", "texture from Gram matrices only"])

## Notes
- Nothing is trained: VGG is frozen and the "model" is the image. This idea of a **perceptual loss** (compare images in the feature space of a pretrained network rather than pixel space) outlived style transfer; it is used to train super-resolution models, the autoencoders of latent diffusion (LPIPS, `dit.ipynb`), and as an evaluation metric.
- Why Gram matrices work: matching them is equivalent to minimizing the Maximum Mean Discrepancy between the two images' feature distributions with a second-order polynomial kernel (Li et al., 2017). Style = the *distribution* of local features, not their arrangement.
- VGG works noticeably better than ResNets for this, likely because it has no skip connections and its features are less sparse.
- It is slow: every image is its own optimization. Follow-ups amortize it: a feed-forward network trained per style with this same loss (Johnson et al., 2016), then **AdaIN** (Huang & Belongie, 2017), which transfers any style in one pass by matching the mean and variance of feature channels, an idea StyleGAN later built on (`gan.ipynb`).
- Today the same effect is usually obtained from text-to-image diffusion models (a style prompt, a style LoRA as in `lora.ipynb`, or an image adapter).
- Common additions: a total-variation loss on $x$ to remove high-frequency noise, and preserving the content image's colors by transferring style only in the luminance channel.